# 06 - Join gnomAD AF + CADD PHRED annotations

Notebook nay tao annotation manh hon cho XGBoost ma khong can VEP cache lon:

- `gnomAD_AF`: allele frequency trong gnomAD
- `CADD_PHRED`: deleteriousness score tu CADD precomputed scores

Notebook khong tu tai file genome-wide rat lon. Thay vao do:

1. Tao query variants tu `processed/clinvar_training_metadata.parquet`.
2. Doc cac file da loc san neu ban dat vao `Data/annotations/`.
3. Merge thanh `Data/annotations/variant_annotations.parquet` va `.csv` de notebook 04 tu dung.

Nguon tai tham khao:

- gnomAD v4 GRCh38 sites VCF: https://gnomad.broadinstitute.org/downloads
- gnomAD public data path pattern: `gs://gcp-public-data--gnomad/release/.../vcf/...`
- CADD download: https://cadd.gs.washington.edu/download

In [ ]:
from pathlib import Path
import gzip
import re

import numpy as np
import pandas as pd

PROJECT_DIR = Path("/mnt/MyData/Bioinformatics/Project")
PROCESSED_DIR = PROJECT_DIR / "processed"
ANNOTATION_DIR = PROJECT_DIR / "Data" / "annotations"
ANNOTATION_DIR.mkdir(exist_ok=True)

METADATA_PATH = PROCESSED_DIR / "clinvar_training_metadata.parquet"
VARIANT_QUERY_PARQUET = ANNOTATION_DIR / "clinvar_annotation_query.parquet"
VARIANT_QUERY_TSV = ANNOTATION_DIR / "clinvar_annotation_query.tsv"
VARIANT_QUERY_BED = ANNOTATION_DIR / "clinvar_annotation_query.bed"

GNOMAD_CANDIDATES = [
    ANNOTATION_DIR / "gnomad_af.parquet",
    ANNOTATION_DIR / "gnomad_af.csv",
    ANNOTATION_DIR / "gnomad_af.tsv",
    ANNOTATION_DIR / "gnomad_af.tsv.gz",
    ANNOTATION_DIR / "gnomad_af.vcf",
    ANNOTATION_DIR / "gnomad_af.vcf.gz",
]

CADD_CANDIDATES = [
    ANNOTATION_DIR / "cadd_scores.parquet",
    ANNOTATION_DIR / "cadd_scores.csv",
    ANNOTATION_DIR / "cadd_scores.tsv",
    ANNOTATION_DIR / "cadd_scores.tsv.gz",
]

OUTPUT_PARQUET = ANNOTATION_DIR / "variant_annotations.parquet"
OUTPUT_CSV = ANNOTATION_DIR / "variant_annotations.csv"

if not METADATA_PATH.exists():
    raise FileNotFoundError(METADATA_PATH)

METADATA_PATH, ANNOTATION_DIR

## 1. Tao query variant list

Key join dung cho tat ca annotation:

`Chromosome, PositionVCF, ReferenceAlleleVCF, AlternateAlleleVCF`

Dong thoi tao BED 0-based de sau nay co the dung voi `tabix`/`bcftools` neu can.

In [ ]:
key_cols = [
    "Chromosome",
    "PositionVCF",
    "ReferenceAlleleVCF",
    "AlternateAlleleVCF",
]

query_df = pd.read_parquet(METADATA_PATH, columns=key_cols).copy()
for col in key_cols:
    query_df[col] = query_df[col].astype(str)

valid_snv_mask = (
    query_df["ReferenceAlleleVCF"].str.fullmatch("[ACGT]")
    & query_df["AlternateAlleleVCF"].str.fullmatch("[ACGT]")
)
if not valid_snv_mask.all():
    raise ValueError(f"Co {(~valid_snv_mask).sum():,} variants khong phai SNV A/C/G/T")

query_df = query_df.drop_duplicates(subset=key_cols).copy()
query_df["PositionVCF_int"] = query_df["PositionVCF"].astype(int)
query_df = query_df.sort_values(["Chromosome", "PositionVCF_int", "ReferenceAlleleVCF", "AlternateAlleleVCF"])
query_out_df = query_df[key_cols].copy()
query_out_df.to_parquet(VARIANT_QUERY_PARQUET, index=False, engine="pyarrow")
query_out_df.to_csv(VARIANT_QUERY_TSV, sep="	", index=False)

bed_df = pd.DataFrame({
    "chrom": query_df["Chromosome"],
    "start0": query_df["PositionVCF_int"] - 1,
    "end0": query_df["PositionVCF_int"],
    "name": (
        query_df["Chromosome"] + ":" + query_df["PositionVCF"] + ":"
        + query_df["ReferenceAlleleVCF"] + ":" + query_df["AlternateAlleleVCF"]
    ),
})
bed_df.to_csv(VARIANT_QUERY_BED, sep="	", header=False, index=False)

print(f"Query variants: {len(query_out_df):,}")
print(f"Saved: {VARIANT_QUERY_PARQUET}")
print(f"Saved: {VARIANT_QUERY_TSV}")
print(f"Saved: {VARIANT_QUERY_BED}")
query_out_df.head()

## 2. File input can dat vao Data/annotations

Notebook se tu tim cac file sau:

### gnomAD

Uu tien:

- `Data/annotations/gnomad_af.parquet`
- `Data/annotations/gnomad_af.tsv.gz`
- `Data/annotations/gnomad_af.vcf.gz`

Schema da loc san nen co:

`Chromosome, PositionVCF, ReferenceAlleleVCF, AlternateAlleleVCF, gnomAD_AF`

Co the them `gnomAD_AF_popmax` neu co.

### CADD

Uu tien:

- `Data/annotations/cadd_scores.parquet`
- `Data/annotations/cadd_scores.tsv.gz`

CADD official precomputed TSV thuong co cot:

`#Chrom, Pos, Ref, Alt, RawScore, PHRED`

Notebook se normalize thanh:

`Chromosome, PositionVCF, ReferenceAlleleVCF, AlternateAlleleVCF, CADD_RAW, CADD_PHRED`

In [ ]:
print("gnomAD candidates:")
for path in GNOMAD_CANDIDATES:
    print(path, "exists" if path.exists() else "missing")

print("\nCADD candidates:")
for path in CADD_CANDIDATES:
    print(path, "exists" if path.exists() else "missing")

## 3. Helper functions normalize schema

In [ ]:
def first_existing(paths):
    return next((path for path in paths if path.exists()), None)


def read_table_auto(path):
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    if path.suffix == ".csv":
        return pd.read_csv(path)
    # .tsv or .tsv.gz
    return pd.read_csv(path, sep="\t", comment="#")


def normalize_key_columns(df):
    rename_map = {
        "#Chrom": "Chromosome",
        "Chrom": "Chromosome",
        "CHROM": "Chromosome",
        "chrom": "Chromosome",
        "POS": "PositionVCF",
        "Pos": "PositionVCF",
        "pos": "PositionVCF",
        "position": "PositionVCF",
        "REF": "ReferenceAlleleVCF",
        "Ref": "ReferenceAlleleVCF",
        "ref": "ReferenceAlleleVCF",
        "ALT": "AlternateAlleleVCF",
        "Alt": "AlternateAlleleVCF",
        "alt": "AlternateAlleleVCF",
    }
    df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns}).copy()
    for col in key_cols:
        if col not in df.columns:
            raise ValueError(f"Missing required key column: {col}")
        df[col] = df[col].astype(str)
    return df


def parse_info_af(info_value):
    if pd.isna(info_value):
        return np.nan
    info = str(info_value)
    match = re.search(r"(?:^|;)AF=([^;]+)", info)
    if not match:
        return np.nan
    value = match.group(1).split(",")[0]
    try:
        return float(value)
    except ValueError:
        return np.nan


def read_simple_vcf(path):
    opener = gzip.open if str(path).endswith(".gz") else open
    header = None
    rows = []
    with opener(path, "rt") as handle:
        for line in handle:
            if line.startswith("##"):
                continue
            if line.startswith("#CHROM"):
                header = line.lstrip("#").rstrip("\n").split("\t")
                continue
            if line.startswith("#") or not line.strip():
                continue
            rows.append(line.rstrip("\n").split("\t"))
    if header is None:
        raise ValueError(f"VCF header not found: {path}")
    if not rows:
        return pd.DataFrame(columns=header)
    df = pd.DataFrame(rows, columns=header[:len(rows[0])])
    return df


def normalize_gnomad(path):
    if path.suffix in {".vcf", ".gz"} and ".vcf" in path.name:
        raw = read_simple_vcf(path)
        raw = raw.rename(columns={"CHROM": "Chromosome", "POS": "PositionVCF", "REF": "ReferenceAlleleVCF", "ALT": "AlternateAlleleVCF"})
        raw["gnomAD_AF"] = raw["INFO"].map(parse_info_af)
        cols = key_cols + ["gnomAD_AF"]
        return normalize_key_columns(raw[cols])

    raw = read_table_auto(path)
    raw = normalize_key_columns(raw)
    if "gnomAD_AF" not in raw.columns:
        for candidate in ["AF", "af", "AF_total", "freq"]:
            if candidate in raw.columns:
                raw = raw.rename(columns={candidate: "gnomAD_AF"})
                break
    if "gnomAD_AF" not in raw.columns:
        raise ValueError("gnomAD file must contain gnomAD_AF or AF column")
    keep = key_cols + ["gnomAD_AF"]
    if "gnomAD_AF_popmax" in raw.columns:
        keep.append("gnomAD_AF_popmax")
    out = raw[keep].copy()
    out["gnomAD_AF"] = pd.to_numeric(out["gnomAD_AF"], errors="coerce")
    if "gnomAD_AF_popmax" in out.columns:
        out["gnomAD_AF_popmax"] = pd.to_numeric(out["gnomAD_AF_popmax"], errors="coerce")
    return out.drop_duplicates(subset=key_cols, keep="first")


def normalize_cadd(path):
    raw = read_table_auto(path)
    raw = normalize_key_columns(raw)
    rename_map = {}
    if "RawScore" in raw.columns:
        rename_map["RawScore"] = "CADD_RAW"
    if "PHRED" in raw.columns:
        rename_map["PHRED"] = "CADD_PHRED"
    raw = raw.rename(columns=rename_map)
    if "CADD_PHRED" not in raw.columns:
        raise ValueError("CADD file must contain CADD_PHRED or PHRED column")
    keep = key_cols + ["CADD_PHRED"]
    if "CADD_RAW" in raw.columns:
        keep.append("CADD_RAW")
    out = raw[keep].copy()
    out["CADD_PHRED"] = pd.to_numeric(out["CADD_PHRED"], errors="coerce")
    if "CADD_RAW" in out.columns:
        out["CADD_RAW"] = pd.to_numeric(out["CADD_RAW"], errors="coerce")
    return out.drop_duplicates(subset=key_cols, keep="first")

## 4. Load gnomAD AF neu co

In [ ]:
gnomad_path = first_existing(GNOMAD_CANDIDATES)
if gnomad_path is None:
    print("Chua co gnomAD AF file. Se tao cot gnomAD_AF rong.")
    gnomad_df = query_out_df.copy()
    gnomad_df["gnomAD_AF"] = np.nan
    gnomad_df["gnomAD_AF_popmax"] = np.nan
else:
    print(f"Loading gnomAD: {gnomad_path}")
    gnomad_df = normalize_gnomad(gnomad_path)

print(gnomad_df.shape)
print("gnomAD_AF non-null:", int(gnomad_df["gnomAD_AF"].notna().sum()))
gnomad_df.head()

## 5. Load CADD PHRED neu co

In [ ]:
cadd_path = first_existing(CADD_CANDIDATES)
if cadd_path is None:
    print("Chua co CADD score file. Se tao cot CADD_PHRED rong.")
    cadd_df = query_out_df.copy()
    cadd_df["CADD_RAW"] = np.nan
    cadd_df["CADD_PHRED"] = np.nan
else:
    print(f"Loading CADD: {cadd_path}")
    cadd_df = normalize_cadd(cadd_path)

print(cadd_df.shape)
print("CADD_PHRED non-null:", int(cadd_df["CADD_PHRED"].notna().sum()))
cadd_df.head()

## 6. Merge annotation va export cho notebook 04

In [ ]:
annotation_df = query_out_df.copy()
annotation_df = annotation_df.merge(gnomad_df, on=key_cols, how="left")
annotation_df = annotation_df.merge(cadd_df, on=key_cols, how="left")

# Keep schema expected by notebook 04. Consequence/SpliceAI remain placeholders for now.
if "consequence" not in annotation_df.columns:
    annotation_df["consequence"] = "unknown"
if "SpliceAI_max" not in annotation_df.columns:
    annotation_df["SpliceAI_max"] = np.nan
if "gnomAD_AF" not in annotation_df.columns:
    annotation_df["gnomAD_AF"] = np.nan
if "CADD_PHRED" not in annotation_df.columns:
    annotation_df["CADD_PHRED"] = np.nan

ordered_cols = key_cols + [
    "consequence",
    "gnomAD_AF",
    "gnomAD_AF_popmax",
    "CADD_RAW",
    "CADD_PHRED",
    "SpliceAI_max",
]
for col in ordered_cols:
    if col not in annotation_df.columns:
        annotation_df[col] = np.nan
annotation_df = annotation_df[ordered_cols]

annotation_df.to_parquet(OUTPUT_PARQUET, index=False, engine="pyarrow")
annotation_df.to_csv(OUTPUT_CSV, index=False)

print(f"Saved: {OUTPUT_PARQUET}")
print(f"Saved: {OUTPUT_CSV}")
print(annotation_df.shape)
print({
    "gnomAD_AF_non_null": int(annotation_df["gnomAD_AF"].notna().sum()),
    "CADD_PHRED_non_null": int(annotation_df["CADD_PHRED"].notna().sum()),
})
annotation_df.head()

## 7. Gợi ý cách tạo file gnomAD/CADD đã lọc

Notebook này đã tạo:

- `Data/annotations/clinvar_annotation_query.tsv`
- `Data/annotations/clinvar_annotation_query.bed`

Nếu có file genome-wide đã tải sẵn, nên lọc bên ngoài thành file nhỏ rồi đặt vào:

- `Data/annotations/gnomad_af.tsv.gz`
- `Data/annotations/cadd_scores.tsv.gz`

Không nên load trực tiếp CADD 81GB hay gnomAD VCF hàng chục GB trong pandas.